In [2]:
import threading, time, math, os, platform
import numpy as np, pandas as pd
from PIL import Image

image = np.array(Image.open("test_images/test_images/large_1024x1024.png").convert("L"), dtype=np.float64)
gaussian_kernel = np.array([[1,2,1],[2,4,2],[1,2,1]], dtype=np.float64) / 16

WORKERS = [1, 2, 4, 8]
REPEATS = 5                     # timed runs (plus 1 discarded warm-up)
COUNTER_UPDATES = 100_000       # fixed total work
PIXEL_REPEATS = 1000

# 9 kernel products at the center pixel; sum(terms) = 18.0625 (matches the group's 18062.5 / 1000)
ci, cj = image.shape[0] // 2, image.shape[1] // 2
terms = [image[ci+k-1, cj+l-1] * gaussian_kernel[k, l] for k in range(3) for l in range(3)]

EXPERIMENTS = {
    "pixel":   terms * PIXEL_REPEATS,     # 9000 updates
    "counter": [1] * COUNTER_UPDATES,     # 100000 updates
}

# ---- workers: each one applies its share of the updates to the shared value ----
def w_plain(shared, amounts, lock, idx):
    for a in amounts:
        shared[0] += a

def w_unsafe(shared, amounts, lock, idx):          # race: read, yield, write
    for a in amounts:
        tmp = shared[0]; time.sleep(0); shared[0] = tmp + a

def w_lock_yield(shared, amounts, lock, idx):      # lock + same yield as unsafe (fair comparison)
    for a in amounts:
        with lock:
            tmp = shared[0]; time.sleep(0); shared[0] = tmp + a

def w_lock(shared, amounts, lock, idx):
    for a in amounts:
        with lock:
            shared[0] += a

def w_local(shared, amounts, lock, idx):           # local accumulation, reduce after join
    acc = 0.0
    for a in amounts:
        acc += a
    shared[idx] = acc

VERSIONS = {
    "sequential":       dict(worker=w_plain,      threaded=False),
    "sequential_yield": dict(worker=w_unsafe,     threaded=False),  # baseline for unsafe/lock_yield
    "unsafe":           dict(worker=w_unsafe,     threaded=True),
    "lock_yield":       dict(worker=w_lock_yield, threaded=True),
    "lock":             dict(worker=w_lock,       threaded=True),
    "local":            dict(worker=w_local,      threaded=True, local=True),
}

def timed(version, updates, n):
    cfg = VERSIONS[version]
    local = cfg.get("local", False)
    shared = [0.0] * n if local else [0.0]
    lock = threading.Lock()
    if not cfg["threaded"]:
        start = time.perf_counter()
        cfg["worker"](shared, updates, lock, 0)
        elapsed = time.perf_counter() - start
        return shared[0], elapsed
    chunks = [updates[w::n] for w in range(n)]     # split outside the timer
    threads = [threading.Thread(target=cfg["worker"], args=(shared, chunks[w], lock, w))
               for w in range(n)]
    start = time.perf_counter()
    for t in threads: t.start()
    for t in threads: t.join()
    actual = sum(shared) if local else shared[0]   # reduction included in timing
    elapsed = time.perf_counter() - start
    return actual, elapsed

print("CPU cores:", os.cpu_count(), "|", platform.processor(), "|", platform.platform())

rows = []
for exp, updates in EXPERIMENTS.items():
    expected = sum(updates)
    for version, cfg in VERSIONS.items():
        for n in (WORKERS if cfg["threaded"] else [1]):
            for run in range(REPEATS + 1):
                actual, t = timed(version, updates, n)
                if run == 0:
                    continue                       # discard warm-up
                rows.append(dict(experiment=exp, version=version, workers=n, run=run,
                                 time_s=t, expected=expected, actual=actual,
                                 output_ok=math.isclose(actual, expected, rel_tol=1e-9)))
            print(f"done: {exp} / {version} / {n} workers")

df = pd.DataFrame(rows)
df.to_csv("timing_results.csv", index=False)

CPU cores: 16 | Intel64 Family 6 Model 186 Stepping 2, GenuineIntel | Windows-11-10.0.26300-SP0
done: pixel / sequential / 1 workers
done: pixel / sequential_yield / 1 workers
done: pixel / unsafe / 1 workers
done: pixel / unsafe / 2 workers
done: pixel / unsafe / 4 workers
done: pixel / unsafe / 8 workers
done: pixel / lock_yield / 1 workers
done: pixel / lock_yield / 2 workers
done: pixel / lock_yield / 4 workers
done: pixel / lock_yield / 8 workers
done: pixel / lock / 1 workers
done: pixel / lock / 2 workers
done: pixel / lock / 4 workers
done: pixel / lock / 8 workers
done: pixel / local / 1 workers
done: pixel / local / 2 workers
done: pixel / local / 4 workers
done: pixel / local / 8 workers
done: counter / sequential / 1 workers
done: counter / sequential_yield / 1 workers
done: counter / unsafe / 1 workers
done: counter / unsafe / 2 workers
done: counter / unsafe / 4 workers
done: counter / unsafe / 8 workers
done: counter / lock_yield / 1 workers
done: counter / lock_yield / 

In [3]:
summary = (df.groupby(["experiment", "version", "workers"])
             .agg(mean_s=("time_s", "mean"), std_s=("time_s", "std"), min_s=("time_s", "min"),
                  correct_runs=("output_ok", "sum"), runs=("output_ok", "count"))
             .reset_index())

for base in ["sequential", "sequential_yield"]:
    ref = summary[summary.version == base].set_index("experiment")["mean_s"]
    summary[f"speedup_vs_{base}"] = summary["experiment"].map(ref) / summary["mean_s"]

summary.to_csv("timing_summary.csv", index=False)
summary

,experiment,version,workers,mean_s,std_s,min_s,correct_runs,runs,speedup_vs_sequential,speedup_vs_sequential_yield
0,counter,local,1,0.004031,0.000095,0.003875,5,5,1.424243,4.844750
1,counter,local,2,0.003981,0.000081,0.003896,5,5,1.442082,4.905431
2,counter,local,4,0.004214,0.000143,0.004096,5,5,1.362414,4.634430
3,counter,local,8,0.004911,0.000072,0.004809,5,5,1.168883,3.976108
4,counter,lock,1,0.022812,0.001887,0.020669,5,5,0.251653,0.856030
5,counter,lock,2,0.021606,0.001944,0.020167,5,5,0.265694,0.903792
6,counter,lock,4,0.021565,0.000792,0.020837,5,5,0.266200,0.905513
7,counter,lock,8,0.021922,0.000292,0.021660,5,5,0.261871,0.890787
8,counter,lock_yield,1,0.038187,0.006121,0.034527,5,5,0.150328,0.511360
9,counter,lock_yield,2,0.080588,0.006474,0.074194,5,5,0.071235,0.242314
